# MEND AI · 최종 모델 웹 + 평가 서버

**1 → 4번 셀을 순서대로 실행하세요.**

- 메뉴 **런타임 → 런타임 유형 변경 → GPU**를 선택합니다. L4/A100은 BF16, T4는 FP32로 실행합니다.
- `mendai_final_bundle.zip`(약 525MB)을 **Google Drive 내 드라이브 최상위**에 올립니다.
- ngrok 토큰은 Colab 왼쪽 열쇠 메뉴의 `NGROK_AUTHTOKEN` 비밀 키 또는 마지막 셀의 비공개 입력창에 넣습니다.
- 마지막 셀이 출력하는 **교수님 제출 주소**를 제출하세요. 웹사이트와 `/health`, `/predict`가 모두 같은 주소에서 열립니다.

DINOv3 EX-06 LoRA+TTA 50% + RF-DETR-S LoRA 1024 원본 25% + YOLO26m A3 TTA 25%. 안감 임계값 0.54 / 밑창 0.40. 새 학습은 하지 않습니다. 평가 중에는 Colab 런타임을 켜 두세요.


## 1. 코드와 패키지 준비


In [ ]:
import os, sys, subprocess
from pathlib import Path
REPO = Path('/content/Mendi_UI')
if not REPO.exists():
    subprocess.check_call(['git','clone','--depth','1','https://github.com/rkdworn/Mendi_UI.git',str(REPO)])
else:
    subprocess.check_call(['git','-C',str(REPO),'pull','--ff-only'])
subprocess.check_call([sys.executable,'-m','pip','install','-q','-r',str(REPO/'requirements.txt')])
# This ensemble does not use torchao quantization. Remove incompatible Colab defaults.
subprocess.check_call([sys.executable,'-m','pip','uninstall','-y','torchao'])
# Test LoRA injection in a fresh process, just like the API server.
subprocess.check_call([sys.executable,'-c',
    'import torch; from peft import LoraConfig, get_peft_model; '
    'get_peft_model(torch.nn.Sequential(torch.nn.Linear(4,4)), LoraConfig(r=2,target_modules=["0"])); '
    'print("PEFT LoRA preflight passed")'])
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime and run again.'
print('GPU:', torch.cuda.get_device_name(0), '| torch:', torch.__version__)
print('Packages ready')


## 2. Drive 연결 및 가중치 묶음 풀기

Drive의 내 드라이브 최상위에 ZIP을 올렸다면 경로를 수정할 필요가 없습니다.


In [ ]:
from google.colab import drive
import zipfile, shutil

drive.mount('/content/drive')
DRIVE_ZIP = Path('/content/drive/MyDrive/mendai_final_bundle.zip')
assert DRIVE_ZIP.is_file(), 'Upload mendai_final_bundle.zip to the root of My Drive first.'
LOCAL_ZIP = Path('/content/mendai_final_bundle.zip')
shutil.copyfile(DRIVE_ZIP, LOCAL_ZIP)
with zipfile.ZipFile(LOCAL_ZIP) as archive:
    for entry in archive.infolist():
        (Path('/content')/entry.filename).resolve().relative_to(Path('/content'))
    archive.extractall('/content')
BUNDLE = Path('/content/mendai_final_bundle')
assert (BUNDLE/'manifest.json').is_file()
print('Model bundle extracted. Model hashes will be checked at startup.')


## 3. 모델 서버 실행

모델 해시를 확인한 뒤 웹 화면과 API를 함께 시작합니다.


In [ ]:
import time, requests
LOG_PATH = Path('/content/mendai_server.log')
if 'server_process' in globals() and server_process.poll() is None:
    server_process.terminate()
    server_process.wait(timeout=20)
log_stream = LOG_PATH.open('w')
server_process = subprocess.Popen(
    [sys.executable,'-m','inference.run','--bundle',str(BUNDLE),'--host','127.0.0.1',
     '--port','8000','--memory-budget-mib','12288'],
    cwd=REPO, stdout=log_stream, stderr=subprocess.STDOUT)
for attempt in range(120):
    if server_process.poll() is not None:
        print(LOG_PATH.read_text()[-12000:])
        raise RuntimeError('Server startup failed. Check the log above.')
    try:
        health = requests.get('http://127.0.0.1:8000/health',timeout=3)
        if health.ok and health.json().get('status')=='ok':
            print('Three models ready:',health.json()['device'])
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    print(LOG_PATH.read_text()[-12000:])
    raise TimeoutError('Server startup timed out.')
assert requests.get('http://127.0.0.1:8000/',timeout=10).ok
print('Website and evaluation API are ready.')


## 4. 공개 주소 만들기 — 이 주소를 제출하세요

고정 도메인이 있다면 Colab 비밀 키 `NGROK_DOMAIN`에도 저장하세요. 없으면 ngrok이 발급한 주소를 사용합니다. 토큰은 출력하지 않습니다.


In [ ]:
from google.colab import userdata
from getpass import getpass
from pyngrok import ngrok
from urllib.parse import quote
from IPython.display import display, HTML

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

token = secret('NGROK_AUTHTOKEN') or getpass('ngrok authtoken (비공개 입력): ')
assert token.strip(), 'ngrok authtoken이 필요합니다.'
ngrok.set_auth_token(token.strip())
del token
if 'tunnel' in globals():
    try:
        ngrok.disconnect(tunnel.public_url)
    except Exception:
        pass
domain = secret('NGROK_DOMAIN')
options = {'domain':domain.strip()} if domain else {}
tunnel = ngrok.connect(addr=8000,proto='http',**options)
PUBLIC_URL = tunnel.public_url
check = requests.get(PUBLIC_URL+'/health',headers={'ngrok-skip-browser-warning':'true'},timeout=30)
check.raise_for_status()
assert check.json()['status']=='ok'
print('교수님 제출 주소 (웹사이트 + 평가 서버):',PUBLIC_URL)
print('서버 점검:',PUBLIC_URL+'/health')
print('교수님 평가 프로그램에도 같은 API 주소를 입력하세요.')
print('GitHub Pages 화면:', 'https://rkdworn.github.io/Mendi_UI/?api='+quote(PUBLIC_URL,safe=''))
display(HTML('<a href="'+PUBLIC_URL+'" target="_blank" style="font-size:22px">신발 손상 분석 웹사이트 열기 ↗</a>'))


## 평가 종료 후

아래 셀은 평가가 끝난 뒤에만 실행하세요. Colab 런타임이 꺼지면 서버도 멈춥니다. 로컬 RTX 5060에서 val 1,319장의 판정과 균형 정확도 88.19%·전체 정확도 86.81%를 재현했습니다. Colab GPU와 torch 버전, T4 FP32 경로에서는 점수가 달라질 수 있습니다. 비중과 임계값은 고정합니다.


In [ ]:
# Run only after evaluation has finished.
if 'tunnel' in globals():
    ngrok.disconnect(tunnel.public_url)
if 'server_process' in globals() and server_process.poll() is None:
    server_process.terminate()
print('Server stopped.')
